In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

✅ Libraries imported successfully!


In [3]:
df = pd.read_csv('dataset.csv')

print('📊 Dataset Shape:', df.shape)
print('Total Songs:', len(df))
print('Total Genres:', df['track_genre'].nunique())
df.head()

📊 Dataset Shape: (114000, 21)
Total Songs: 114000
Total Genres: 114


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,...,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,...,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,...,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,...,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,...,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,...,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
# Duplicates hatao
df = df.drop_duplicates(subset=['track_name', 'artists']).reset_index(drop=True)
df = df.drop(columns=['Unnamed: 0'], errors='ignore')

print(f'✅ Songs after cleaning: {len(df):,}')

# Missing values check
print(f'Missing values: {df.isnull().sum().sum()}')

# Audio features select karo
AUDIO_FEATURES = ['danceability', 'energy', 'loudness', 'speechiness',
                  'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Normalize karo (0 to 1)
scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[AUDIO_FEATURES] = scaler.fit_transform(df[AUDIO_FEATURES])

print('✅ Data normalized successfully!')
df_scaled[AUDIO_FEATURES].describe()

✅ Songs after cleaning: 81,344
Missing values: 3
✅ Data normalized successfully!


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
count,81344.000000,81344.000000,81344.000000,81344.000000,81344.000000,81344.000000,81344.000000,81344.000000,81344.000000
mean,0.567792,0.635025,0.757210,0.092220,0.330994,0.184731,0.219721,0.465608,0.501886
std,0.180452,0.258639,0.098122,0.120858,0.341326,0.331591,0.198271,0.264707,0.123798
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.452792,0.455000,0.722856,0.037409,0.015964,0.000000,0.098500,0.242211,0.408461
50%,0.581726,0.678000,0.781847,0.050881,0.190763,0.000089,0.133000,0.451256,0.501413
75%,0.700508,0.857000,0.821098,0.090155,0.631526,0.153000,0.283000,0.679397,0.575778
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [5]:
# Feature matrix banao
feature_matrix = df_scaled[AUDIO_FEATURES].values

# KNN Model train karo
knn_model = NearestNeighbors(n_neighbors=11, metric='cosine', algorithm='brute')
knn_model.fit(feature_matrix)

print(f'✅ KNN Model trained on {len(df_scaled):,} songs!')

✅ KNN Model trained on 81,344 songs!


In [6]:
def recommend_songs(song_name, n_recommendations=5):
    # Song dhoondo dataset mein
    mask = df['track_name'].str.lower() == song_name.lower()
    matches = df[mask]
    
    if matches.empty:
        print(f'❌ Song "{song_name}" nahi mila!')
        return
    
    song_idx = matches.index[0]
    song_info = df.loc[song_idx]
    print(f'🎵 Song mila: {song_info["track_name"]} — {song_info["artists"]}')
    print('-' * 50)
    
    # Similar songs dhoondo
    song_vector = feature_matrix[song_idx].reshape(1, -1)
    distances, indices = knn_model.kneighbors(song_vector, n_neighbors=n_recommendations+1)
    
    print(f'\n🎧 Top {n_recommendations} Recommendations:\n')
    for i, (idx, dist) in enumerate(zip(indices[0][1:], distances[0][1:]), 1):
        rec = df.loc[idx]
        similarity = round((1 - dist) * 100, 1)
        print(f'{i}. {rec["track_name"]} — {rec["artists"]} | Similarity: {similarity}%')

# Test karo!
recommend_songs('Starboy')


🎵 Song mila: Starboy — The Weeknd;Daft Punk
--------------------------------------------------

🎧 Top 5 Recommendations:

1. Benden Uzak — Tanerman;Lia Shine;Yung Ouzo | Similarity: 99.8%
2. Love Letters Intro / Love From The Sick Side — The Psycho Realm | Similarity: 99.7%
3. overwhelmed - Chri$tian Gate$ remix — Royal & the Serpent;Chri$tian Gate$ | Similarity: 99.7%
4. LA CANCIÓN — J Balvin;Bad Bunny | Similarity: 99.7%
5. Don't Like — Goldy;Karan Aujla | Similarity: 99.7%
